
### IVOIRE Project — Review Sentiment Analysis
Multilingual sentiment scoring (1-5 stars) on Portuguese review comments,
compared against the existing `review_score`. Output re-imported to
PostgreSQL as `review_sentiment`, linked by (`review_id`, `order_id`).

In [16]:
import pandas as pd
from sqlalchemy import create_engine
from transformers import pipeline

### 1. Connection & extraction
Only rows with an actual comment are pulled — most reviews are score-only.

In [ ]:
engine = create_engine(
    "postgresql+psycopg2://postgres:@localhost:5432/Brazilian E-Commerce [IVOIRE]"
)

In [18]:
query = ("""
    SELECT review_id, order_id, review_score, review_comment_message
    FROM fact_reviews
    WHERE review_comment_message IS NOT NULL;
""")

In [19]:
df = pd.read_sql(query, engine)

### 2. Load the sentiment model
Multilingual model, no translation step — handles Portuguese directly.
Output label format is "X stars", matching review_score's 1-5 scale.

In [20]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8037.59it/s]


### 3. Run inference
Batched (not one-by-one) for speed. truncation=True + max_length=512
handles comments longer than the model's token limit — BERT-based models
can't accept sequences beyond that, so anything longer gets cut rather
than raising an error.

In [21]:
texts = df["review_comment_message"].tolist()

In [22]:
results = sentiment_pipeline(
    texts,
    batch_size=32,
    truncation=True,
    max_length=512
)

### 4. Parse the model output
Label comes back as "X stars" (string) — extract the leading digit as int.
Confidence score kept alongside, useful to filter out low-confidence
predictions later if needed.

In [23]:
df["predicted_sentiment_score"] = [int(r["label"][0]) for r in results]
df["prediction_confidence"] = [r["score"] for r in results]

### 5. Compare against review_score
Positive gap = comment text reads more positive than the star rating given.

Negative gap = comment text reads more negative than the star rating given.

Large gaps (either direction) are worth spot-checking individually.

In [25]:
df["sentiment_score_gap"] = df["predicted_sentiment_score"] - df["review_score"]

### 6. Re-import to PostgreSQL

In [26]:
review_sentiment = df[[
    "review_id",
    "order_id",
    "review_score",
    "predicted_sentiment_score",
    "prediction_confidence",
    "sentiment_score_gap",
]].copy()
 
review_sentiment.to_sql("review_sentiment", engine, if_exists="replace", index=False)
 
print("review_sentiment written to PostgreSQL.")

review_sentiment written to PostgreSQL.
